# Figure 6: do the final chains agree?

The final model is fitted as several independent chains with the calibrated settings (`FINAL_CHAINS`); chain 1 is the model of Figures 4 and 5.  Every type of every chain is matched one-to-one to a type of the medoid chain, by the cosine similarity of $\hat\phi_k$, and all the type vectors are embedded together by $t$-SNE.  Colour gives the matched type, bubble area the type's usage; the medoid chain's bubbles have a dark edge.  If every chain found the same solution, the bubbles form $K$ tight clusters of one colour each, one bubble per chain.  A cluster of mixed colours, or a bubble far from the others of its colour, marks a chain that found something else.  With a few dozen points the layout is illustration only: the table of matched cosines is the quantitative check.

Input: `RESULTS/final/K*-c*/` (`blat run`).

**Where to start Jupyter.**  Start it from the top directory (`jupyter lab`), so the file
browser shows the notebooks and the results side by side.  Starting it anywhere else
works too: Jupyter always runs a notebook in the notebook's own folder, and the first
cell finds the top directory from there.

**Which results.**  Set `RESULTS` in the first cell to the results directory of your run,
relative to the top directory: `example/results-quick` (the default) or
`example/results-thorough`.  When `run_final.sh` executes the notebooks it sets it for
you, through the environment variable `BLAT_RESULTS`.

In [ ]:
import os, sys
from pathlib import Path

# the top directory: the nearest folder, from here upwards, that holds the blat package
TOP = next(p for p in [Path.cwd(), *Path.cwd().parents]
                      if (p / "blat" / "__init__.py").exists())
sys.path.insert(0, str(TOP))

# the results to draw, relative to the top directory (or absolute)
RESULTS = TOP / os.environ.get("BLAT_RESULTS", "example/results-quick")
FIGURES = RESULTS / "figures"
FIGURES.mkdir(parents=True, exist_ok=True)
print("results:", RESULTS)

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from blat import figures as F

def save(fig, name):
    for suffix in ("pdf", "png"):
        fig.savefig(FIGURES / f"{name}.{suffix}", dpi=300, bbox_inches="tight")
    print("wrote", FIGURES / f"{name}.pdf")

In [ ]:
finals = F.load_runs(RESULTS / "final")
K = min(finals)
runs = finals[K]
print("K =", K, "|", len(runs), "chains:", ", ".join(r.name for r in runs))

Per chain: the worst and the mean cosine between its types and their matches in the medoid chain (1 = identical).

In [ ]:
vectors, labels = F.chain_type_vectors(runs)
labels.groupby("chain")["cosine"].agg(["min", "mean"]).round(3)

The $t$-SNE settings.  The perplexity is roughly the number of neighbours each point attends to; it must be below the number of type vectors.  `None` uses $\min(10, (n-1)/3)$ for $n$ vectors.  The layout also depends on the seed.

In [ ]:
PERPLEXITY = None      # e.g. 5; None for the default
SEED = 0
print(len(vectors), "type vectors; perplexity",
      PERPLEXITY or F.default_perplexity(len(vectors)))

In [ ]:
points = F.embed(vectors, seed=SEED, perplexity=PERPLEXITY)
fig = F.plot_chain_embedding(points, labels)
save(fig, f"figure6-chains-K{K}")